<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/LSTM-model/LSTM_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduzione

Carichiamo tutto il necessario:

il progetto GitHub
le librerie utili
il file zip del dataset da Google Drive, di cui facciamo l'unzip

In [8]:
!git clone --branch dev --single-branch https://github.com/dokunoale/chagas.git
!pip install wfdb
!pip install -q gdown
from google.colab import drive
drive.mount('/content/drive')

import gdown
import numpy as np
import matplotlib.pyplot as plt
import wfdb
import importlib

import sys
sys.path.append('/content/chagas/src')
from preprocessing import tf_dataset_loader

# Importa tutto dai moduli
from models import libraries, losses, metrics, confusion
from models.LSTM import build_lstm_blocl, classifier

url = "https://drive.google.com/uc?id=1P3blYg3gBMluOazE69ivQFyzHs7tSeq_"
gdown.download(url, "dataset.zip", quiet=False, fuzzy=True)
!unzip -q dataset.zip -d ./dataset

fatal: destination path 'chagas' already exists and is not an empty directory.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ModuleNotFoundError: No module named 'models'

In [2]:
!cd chagas && git checkout dev && git pull origin dev

Already on 'dev'
Your branch is up to date with 'origin/dev'.
From https://github.com/dokunoale/chagas
 * branch            dev        -> FETCH_HEAD
Already up to date.


A questo punto, grazie alla funzione tf_dataset_loader presente su GitHub carichiamo il dataset.

In [3]:
#Aggiorniamo tf_dataset_loader
importlib.reload(tf_dataset_loader)

# Carichiamo il dataset
X_pos, y_pos = tf_dataset_loader.load_dataset('/content/dataset/preprocessed/positives')
X_neg, y_neg = tf_dataset_loader.load_dataset('/content/dataset/preprocessed/negatives')

In [4]:
#Uniamo i positivi e i negativi
X = np.concatenate([X_pos, X_neg], axis=0)
y = np.concatenate([y_pos, y_neg], axis=0)

# Facciamo lo shuffle
indices = np.arange(X.shape[0])
np.random.shuffle(indices)

# Riordinamento di X e y con gli stessi indici
X = X[indices]
y = y[indices]

In [5]:
print(X.shape)

(7131, 2800, 12)


In [6]:
#SPLIT DEL DATASET
from sklearn.model_selection import train_test_split

# Prima train+val / test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Poi train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
# 0.25 * 0.8 = 0.2 del totale -> val 20%, train 60%

In [7]:
# mettiamo le etichette sottoforma di 0 e 1
y_train = y_train.astype(int)
y_test = y_test.astype(int)

print(np.unique(y_train))

[0 1]


# Modello 1: BCE

In [ ]:
model1 =